# 02 质量验证示例

本 Notebook 比较 ArcGIS 预处理后的道路输入与道路清洗结果，并从几何、短线、二维拓扑、连通性、悬挂端点、建筑穿越、缺口候选和运行稳定性等维度生成质量报告。

运行前应先执行 `01_道路清洗示例.ipynb`。反事实建筑穿越审计 CSV 为可选输入；未提供时，仍会依据清洗前后道路直接计算最终残余新增建筑穿越。

In [ ]:
# 运行环境检查：仅检查，不自动安装
import importlib.util

REQUIRED_MODULES = ['numpy', 'pandas', 'geopandas', 'shapely', 'pyarrow', 'matplotlib']

missing_modules = [
    module
    for module in REQUIRED_MODULES
    if importlib.util.find_spec(module) is None
]

if missing_modules:
    raise ImportError(
        "当前环境缺少以下依赖："
        + ", ".join(missing_modules)
        + "\n可在终端或新的 Notebook Cell 中安装："
        + "\npython -m pip install numpy pandas geopandas shapely pyarrow matplotlib scipy pyogrio"
    )

print("依赖检查通过。")


## 1. 环境、路径与评价参数

In [ ]:
# ============================================================
# 广安门内道路清洗 V13｜全维度质量验证
#
# 评价对象：
#   处理前：ArcGIS Integrate + Dissolve 后的输入路网
#   处理后：V13 streets_0.parquet
#
# 评价维度：
#   1. 几何有效性
#   2. 重复、重叠与短线
#   3. 网络规模
#   4. 节点度数结构
#   5. 连通分量
#   6. 悬挂节点与近距离缺口
#   7. 建筑穿越保护
#   8. 缺口修复提案
#   9. 运行稳定性与资源消耗
#
# 输出：
#   - 全指标长表 CSV
#   - 处理前后比较表 CSV
#   - 过程控制指标 CSV
#   - 节点度数分布 CSV
#   - 连通分量表 CSV
#   - 质量问题空间图层 GPKG
#   - 指标汇总 TXT
#   - 指标仪表板 PNG
# ============================================================

import datetime
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

from shapely.geometry import (
    Point,
    LineString,
)

import matplotlib.pyplot as plt
import matplotlib as mpl

from IPython.display import display


# ============================================================
# 0. 环境、字体与警告
# ============================================================

warnings.filterwarnings(
    "ignore",
    message=(
        r"Signature .*numpy\.longdouble.*"
        r"does not match any known type.*"
    ),
    category=UserWarning,
)

mpl.rcParams["font.family"] = "sans-serif"

mpl.rcParams["font.sans-serif"] = [
    "WenQuanYi Micro Hei",
    "WenQuanYi Zen Hei",
    "Microsoft YaHei",
    "SimHei",
    "DejaVu Sans",
]

mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 路径与参数
# ============================================================

TARGET_CRS = "EPSG:4547"
REGION_ID = 0


def locate_project_dir():
    """定位包含样例数据目录的项目根目录。"""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "03_小范围样例数据").exists():
            return candidate
    if cwd.name == "02_精简版代码":
        return cwd.parent
    return cwd

PROJECT_DIR = locate_project_dir()
DATA_DIR = PROJECT_DIR / "03_小范围样例数据"
OUTPUT_ROOT = PROJECT_DIR / "04_示例输出"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REGION_HULL_PATH = DATA_DIR / "boundary_sample.parquet"
BEFORE_STREETS_PATH = DATA_DIR / "streets_before_sample.parquet"
AFTER_STREETS_PATH = OUTPUT_ROOT / "streets_cleaned_sample.parquet"
BUILDINGS_PATH = DATA_DIR / "buildings_sample.parquet"

# 道路清洗 Notebook 的过程输出
V13_PERFORMANCE_PATH = OUTPUT_ROOT / "road_cleaning_sample_performance.csv"
V13_GRID_STATUS_PATH = OUTPUT_ROOT / "road_cleaning_sample_grid_status.csv"
V13_STAGE_AUDIT_PATH = OUTPUT_ROOT / "road_cleaning_sample_stage_audit.csv"
V13_GAP_GPKG_PATH = OUTPUT_ROOT / "road_cleaning_sample_close_gap_candidates.gpkg"

# 可选：完整项目中的反事实建筑穿越审计结果。
# 文件不存在时，Notebook 仍可完成最终残余新增穿越验证。
CROSSING_SUMMARY_PATH = OUTPUT_ROOT / "building_crossing_prevention_summary.csv"
CROSSING_STAGE_PATH = OUTPUT_ROOT / "building_crossing_prevention_by_stage.csv"
CROSSING_GRID_PATH = OUTPUT_ROOT / "building_crossing_prevention_by_grid.csv"

OUTPUT_DIR = OUTPUT_ROOT / "quality_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_COMPARISON_CSV = OUTPUT_DIR / "01_before_after_comparison.csv"
OUTPUT_PROCESS_CSV = OUTPUT_DIR / "02_process_control_metrics.csv"
OUTPUT_ALL_METRICS_CSV = OUTPUT_DIR / "03_all_quality_metrics_long.csv"
OUTPUT_DEGREE_CSV = OUTPUT_DIR / "04_node_degree_distribution.csv"
OUTPUT_COMPONENT_CSV = OUTPUT_DIR / "05_connected_components.csv"
OUTPUT_SUMMARY_WIDE_CSV = OUTPUT_DIR / "06_quality_summary_wide.csv"
OUTPUT_GPKG = OUTPUT_DIR / "07_quality_validation_layers.gpkg"
OUTPUT_REPORT_TXT = OUTPUT_DIR / "08_quality_validation_report.txt"
OUTPUT_DASHBOARD_PNG = OUTPUT_DIR / "09_quality_validation_dashboard.png"


# ------------------------------------------------------------
# 评价参数
# ------------------------------------------------------------

BOUNDARY_EXCLUSION_M = 15.0

BUILDING_INSET_M = 0.30
CROSSING_MATCH_TOLERANCE_M = 0.50

NODE_ROUND_DECIMALS = 3

MIN_TOPOLOGY_EDGE_M = 0.001

SHORT_LINE_THRESHOLD_1_M = 0.50
SHORT_LINE_THRESHOLD_2_M = 5.00
SHORT_LINE_THRESHOLD_3_M = 10.00

SMALL_COMPONENT_THRESHOLD_1_M = 100.0
SMALL_COMPONENT_THRESHOLD_2_M = 500.0

DANGLE_PAIR_MAX_DISTANCE_M = 10.0

GAP_DEDUP_PRECISION_M = 0.001


# ============================================================
# 2. 输入文件检查
# ============================================================

required_paths = [
    REGION_HULL_PATH,
    BEFORE_STREETS_PATH,
    AFTER_STREETS_PATH,
    BUILDINGS_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:

    raise FileNotFoundError(
        "以下必要输入文件不存在：\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ============================================================


## 2. 通用几何与输入整理函数

In [ ]:
# 3. 通用函数
# ============================================================

def make_empty_gdf(
    columns=None,
    crs=TARGET_CRS,
):
    """
    创建带有标准geometry列的空GeoDataFrame。
    """

    columns = columns or {}

    data = {
        name: pd.Series(dtype=dtype)
        for name, dtype in columns.items()
    }

    return gpd.GeoDataFrame(
        data,
        geometry=gpd.GeoSeries(
            [],
            crs=crs,
            name="geometry",
        ),
        crs=crs,
    )


def iter_line_geometries(
    geometry,
):
    """
    从复杂几何中递归提取LineString。
    """

    if (
        geometry is None
        or geometry.is_empty
    ):
        return

    if geometry.geom_type == "LineString":

        if (
            geometry.length > 0
            and len(geometry.coords) >= 2
        ):

            yield geometry

    elif geometry.geom_type in [
        "MultiLineString",
        "GeometryCollection",
    ]:

        for part in geometry.geoms:

            yield from iter_line_geometries(
                part
            )


def iter_polygon_geometries(
    geometry,
):
    """
    从复杂几何中递归提取Polygon。
    """

    if (
        geometry is None
        or geometry.is_empty
    ):
        return

    if geometry.geom_type == "Polygon":

        yield geometry

    elif geometry.geom_type in [
        "MultiPolygon",
        "GeometryCollection",
    ]:

        for part in geometry.geoms:

            yield from iter_polygon_geometries(
                part
            )


def safe_length(
    geometry,
):
    """
    安全读取几何长度。
    """

    if (
        geometry is None
        or geometry.is_empty
    ):

        return 0.0

    return float(
        geometry.length
    )


def extract_line_geometry(
    geometry,
):
    """
    从复杂几何中提取并合并所有线。
    """

    parts = list(
        iter_line_geometries(
            geometry
        )
    )

    if not parts:

        return shapely.GeometryCollection()

    return shapely.union_all(
        parts
    )


def geometry_to_line_gdf(
    geometry,
    minimum_length_m=0.0,
    attributes=None,
):
    """
    将复杂线几何拆分为GeoDataFrame。
    """

    attributes = attributes or {}

    records = []

    for part_id, line in enumerate(
        iter_line_geometries(
            geometry
        )
    ):

        length_m = float(
            line.length
        )

        if length_m <= minimum_length_m:
            continue

        record = {
            "part_id": int(part_id),
            "length_m": length_m,
            "geometry": line,
        }

        record.update(
            attributes
        )

        records.append(
            record
        )

    if not records:

        return make_empty_gdf(
            columns={
                "part_id": "int64",
                "length_m": "float64",
            }
        )

    return gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs=TARGET_CRS,
    )


def normalized_wkb(
    geometries,
    precision_m=None,
):
    """
    标准化几何方向后生成WKB，
    用于正反方向无关的重复判断。
    """

    geometry_array = geometries

    if precision_m is not None:

        geometry_array = shapely.set_precision(
            geometry_array,
            precision_m,
        )

    geometry_array = shapely.normalize(
        geometry_array
    )

    return shapely.to_wkb(
        geometry_array,
        hex=True,
    )


def read_optional_csv(
    path,
):
    """
    可选读取CSV。文件不存在时返回空DataFrame。
    """

    if not path.exists():

        return pd.DataFrame()

    try:

        return pd.read_csv(
            path
        )

    except Exception as error:

        print(
            f"[警告] 无法读取CSV：{path}\n"
            f"{error}"
        )

        return pd.DataFrame()


def first_numeric(
    dataframe,
    column_names,
    default=np.nan,
):
    """
    从候选字段中读取第一项数值。
    """

    if dataframe is None or dataframe.empty:

        return default

    for column in column_names:

        if column in dataframe.columns:

            value = dataframe.iloc[0][
                column
            ]

            try:

                return float(
                    value
                )

            except Exception:

                continue

    return default


def percent_change(
    before,
    after,
):
    """
    计算变化率。
    """

    if (
        before is None
        or not np.isfinite(before)
        or before == 0
    ):

        return np.nan

    return (
        after - before
    ) / before


# ============================================================
# 4. 路网输入整理与几何质量
# ============================================================

def prepare_line_parts(
    source_gdf,
    region_geometry,
    dataset_name,
):
    """
    统一完成：
    1. 原文件几何质量统计；
    2. 有效线要素筛选；
    3. 裁剪至真实评价边界；
    4. 拆分为纯LineString。
    """

    source_gdf = source_gdf.to_crs(
        TARGET_CRS
    ).copy()

    geometry = source_gdf.geometry

    missing_mask = geometry.isna()
    empty_mask = geometry.is_empty

    nonmissing_nonempty = (
        ~missing_mask
        & ~empty_mask
    )

    invalid_mask = (
        nonmissing_nonempty
        & ~geometry.is_valid
    )

    non_line_mask = (
        nonmissing_nonempty
        & ~geometry.geom_type.isin([
            "LineString",
            "MultiLineString",
            "GeometryCollection",
        ])
    )

    has_z_mask = pd.Series(
        False,
        index=source_gdf.index,
    )

    if nonmissing_nonempty.any():

        try:

            has_z_mask.loc[
                nonmissing_nonempty
            ] = shapely.has_z(
                geometry.loc[
                    nonmissing_nonempty
                ].array
            )

        except Exception:

            pass

    valid_source_mask = (
        nonmissing_nonempty
        & geometry.is_valid
        & geometry.geom_type.isin([
            "LineString",
            "MultiLineString",
            "GeometryCollection",
        ])
    )

    valid_source = source_gdf.loc[
        valid_source_mask
    ].copy()

    if valid_source.empty:

        raise RuntimeError(
            f"{dataset_name}没有有效线要素。"
        )

    valid_source["geometry"] = (
        shapely.force_2d(
            valid_source.geometry.array
        )
    )

    intersects_mask = valid_source.intersects(
        region_geometry
    )

    intersecting_features = valid_source.loc[
        intersects_mask
    ].copy()

    clipped = gpd.clip(
        intersecting_features,
        region_geometry,
    )

    line_records = []

    for source_id, geom in enumerate(
        clipped.geometry
    ):

        for line in iter_line_geometries(
            geom
        ):

            line_records.append({
                "source_part_id": source_id,
                "length_m": float(
                    line.length
                ),
                "geometry": line,
            })

    if line_records:

        line_parts = gpd.GeoDataFrame(
            line_records,
            geometry="geometry",
            crs=TARGET_CRS,
        )

    else:

        line_parts = make_empty_gdf(
            columns={
                "source_part_id": "int64",
                "length_m": "float64",
            }
        )

    if line_parts.empty:

        raise RuntimeError(
            f"{dataset_name}裁剪后没有有效线段。"
        )

    line_parts = line_parts[
        line_parts.geometry.length > 0
    ].copy()

    line_parts["length_m"] = (
        line_parts.geometry.length
    )

    # 完全重复：忽略线方向
    line_parts["_normalized_wkb"] = (
        normalized_wkb(
            line_parts.geometry.array
        )
    )

    duplicate_redundant_count = int(
        line_parts[
            "_normalized_wkb"
        ].duplicated(
            keep="first"
        ).sum()
    )

    unique_line_parts = (
        line_parts
        .drop_duplicates(
            subset="_normalized_wkb"
        )
        .drop(
            columns="_normalized_wkb"
        )
        .reset_index(
            drop=True
        )
    )

    # 溶解重叠线后总长度
    line_union = shapely.union_all(
        unique_line_parts.geometry.array
    )

    total_length_m = float(
        line_parts.geometry.length.sum()
    )

    unique_part_length_m = float(
        unique_line_parts.geometry.length.sum()
    )

    union_length_m = safe_length(
        line_union
    )

    overlap_redundant_length_m = max(
        0.0,
        unique_part_length_m
        - union_length_m,
    )

    lengths = line_parts.geometry.length

    non_simple_count = int(
        (
            ~line_parts.geometry.is_simple
        ).sum()
    )

    quality = {
        "dataset": dataset_name,

        "file_feature_count": int(
            len(source_gdf)
        ),

        "missing_geometry_count": int(
            missing_mask.sum()
        ),

        "empty_geometry_count": int(
            empty_mask.sum()
        ),

        "invalid_geometry_count": int(
            invalid_mask.sum()
        ),

        "non_line_geometry_count": int(
            non_line_mask.sum()
        ),

        "has_z_geometry_count": int(
            has_z_mask.sum()
        ),

        "intersecting_source_feature_count": int(
            len(intersecting_features)
        ),

        "line_part_count": int(
            len(line_parts)
        ),

        "unique_line_part_count": int(
            len(unique_line_parts)
        ),

        "exact_duplicate_redundant_count": (
            duplicate_redundant_count
        ),

        "non_simple_line_count": (
            non_simple_count
        ),

        "total_part_length_m": (
            total_length_m
        ),

        "unique_part_length_m": (
            unique_part_length_m
        ),

        "union_length_m": (
            union_length_m
        ),

        "overlap_redundant_length_m": (
            overlap_redundant_length_m
        ),

        "line_count_le_0_5m": int(
            (
                lengths
                <= SHORT_LINE_THRESHOLD_1_M
            ).sum()
        ),

        "line_count_0_5_to_5m": int(
            (
                (lengths > SHORT_LINE_THRESHOLD_1_M)
                & (lengths <= SHORT_LINE_THRESHOLD_2_M)
            ).sum()
        ),

        "line_count_5_to_10m": int(
            (
                (lengths > SHORT_LINE_THRESHOLD_2_M)
                & (lengths <= SHORT_LINE_THRESHOLD_3_M)
            ).sum()
        ),

        "line_count_gt_10m": int(
            (
                lengths
                > SHORT_LINE_THRESHOLD_3_M
            ).sum()
        ),

        "line_length_mean_m": float(
            lengths.mean()
        ),

        "line_length_median_m": float(
            lengths.median()
        ),

        "line_length_p25_m": float(
            lengths.quantile(0.25)
        ),

        "line_length_p75_m": float(
            lengths.quantile(0.75)
        ),

        "line_length_p95_m": float(
            lengths.quantile(0.95)
        ),
    }

    return {
        "quality": quality,
        "line_parts": line_parts,
        "unique_line_parts": unique_line_parts,
        "line_union": line_union,
    }


# ============================================================


## 3. 二维节点—边拓扑与连通分量

In [ ]:
# 5. 网络拓扑
# ============================================================

class DisjointSet:
    """
    简单并查集，用于识别连通分量。
    """

    def __init__(self):

        self.parent = {}
        self.rank = {}

    def add(
        self,
        item,
    ):

        if item not in self.parent:

            self.parent[item] = item
            self.rank[item] = 0

    def find(
        self,
        item,
    ):

        parent = self.parent[
            item
        ]

        if parent != item:

            self.parent[item] = self.find(
                parent
            )

        return self.parent[
            item
        ]

    def union(
        self,
        item_a,
        item_b,
    ):

        self.add(
            item_a
        )

        self.add(
            item_b
        )

        root_a = self.find(
            item_a
        )

        root_b = self.find(
            item_b
        )

        if root_a == root_b:
            return

        rank_a = self.rank[
            root_a
        ]

        rank_b = self.rank[
            root_b
        ]

        if rank_a < rank_b:

            self.parent[
                root_a
            ] = root_b

        elif rank_a > rank_b:

            self.parent[
                root_b
            ] = root_a

        else:

            self.parent[
                root_b
            ] = root_a

            self.rank[
                root_a
            ] += 1


def node_key(
    coordinate,
):
    """
    将端点坐标转换为稳定节点编号。
    """

    return (
        round(
            float(coordinate[0]),
            NODE_ROUND_DECIMALS,
        ),
        round(
            float(coordinate[1]),
            NODE_ROUND_DECIMALS,
        ),
    )


def create_dangle_pair_layer(
    dangle_nodes,
    max_distance_m=DANGLE_PAIR_MAX_DISTANCE_M,
):
    """
    为边界外度为1节点识别最近邻缺口候选。

    每个节点只选择最近的另一个度1节点，
    并对成对结果去重。
    """

    if (
        dangle_nodes is None
        or len(dangle_nodes) < 2
    ):

        return make_empty_gdf(
            columns={
                "pair_id": "int64",
                "distance_m": "float64",
                "distance_class": "object",
            }
        )

    coordinates = np.column_stack([
        dangle_nodes.geometry.x.to_numpy(),
        dangle_nodes.geometry.y.to_numpy(),
    ])

    try:

        from scipy.spatial import cKDTree

        tree = cKDTree(
            coordinates
        )

        distances, neighbours = tree.query(
            coordinates,
            k=2,
        )

        nearest_distances = distances[
            :,
            1
        ]

        nearest_indices = neighbours[
            :,
            1
        ]

    except Exception:

        nearest_distances = np.full(
            len(coordinates),
            np.inf,
        )

        nearest_indices = np.full(
            len(coordinates),
            -1,
            dtype=int,
        )

        for index_a in range(
            len(coordinates)
        ):

            differences = (
                coordinates
                - coordinates[index_a]
            )

            distances = np.sqrt(
                (
                    differences ** 2
                ).sum(
                    axis=1
                )
            )

            distances[
                index_a
            ] = np.inf

            index_b = int(
                np.argmin(
                    distances
                )
            )

            nearest_distances[
                index_a
            ] = distances[
                index_b
            ]

            nearest_indices[
                index_a
            ] = index_b

    pair_keys = set()
    records = []

    for index_a, (
        distance_m,
        index_b,
    ) in enumerate(
        zip(
            nearest_distances,
            nearest_indices,
        )
    ):

        if (
            index_b < 0
            or not np.isfinite(
                distance_m
            )
            or distance_m > max_distance_m
        ):

            continue

        pair_key = tuple(
            sorted([
                int(index_a),
                int(index_b),
            ])
        )

        if pair_key in pair_keys:
            continue

        pair_keys.add(
            pair_key
        )

        point_a = dangle_nodes.geometry.iloc[
            pair_key[0]
        ]

        point_b = dangle_nodes.geometry.iloc[
            pair_key[1]
        ]

        if distance_m <= 0.5:

            distance_class = "≤0.5 m"

        elif distance_m <= 5:

            distance_class = "0.5–5 m"

        else:

            distance_class = "5–10 m"

        records.append({
            "pair_id": len(
                records
            ),
            "node_a": int(
                pair_key[0]
            ),
            "node_b": int(
                pair_key[1]
            ),
            "distance_m": float(
                distance_m
            ),
            "distance_class": (
                distance_class
            ),
            "geometry": LineString([
                point_a,
                point_b,
            ]),
        })

    if not records:

        return make_empty_gdf(
            columns={
                "pair_id": "int64",
                "node_a": "int64",
                "node_b": "int64",
                "distance_m": "float64",
                "distance_class": "object",
            }
        )

    return gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs=TARGET_CRS,
    )


def build_topology_metrics(
    line_union,
    region_geometry,
    dataset_name,
):
    """
    将已溶解路网节点化并建立二维平面拓扑。
    """

    noded_union = shapely.union_all([
        line_union
    ])

    edge_lines = [
        line
        for line in iter_line_geometries(
            noded_union
        )
        if (
            line.length
            > MIN_TOPOLOGY_EDGE_M
        )
    ]

    if not edge_lines:

        raise RuntimeError(
            f"{dataset_name}无法生成拓扑边。"
        )

    disjoint_set = DisjointSet()

    node_degree = {}
    edge_records = []

    for edge_id, line in enumerate(
        edge_lines
    ):

        coordinates = list(
            line.coords
        )

        start_key = node_key(
            coordinates[0]
        )

        end_key = node_key(
            coordinates[-1]
        )

        disjoint_set.add(
            start_key
        )

        disjoint_set.add(
            end_key
        )

        if start_key == end_key:

            node_degree[
                start_key
            ] = (
                node_degree.get(
                    start_key,
                    0,
                )
                + 2
            )

        else:

            node_degree[
                start_key
            ] = (
                node_degree.get(
                    start_key,
                    0,
                )
                + 1
            )

            node_degree[
                end_key
            ] = (
                node_degree.get(
                    end_key,
                    0,
                )
                + 1
            )

            disjoint_set.union(
                start_key,
                end_key,
            )

        edge_records.append({
            "edge_id": int(
                edge_id
            ),
            "start_key": start_key,
            "end_key": end_key,
            "length_m": float(
                line.length
            ),
            "geometry": line,
        })

    roots = {
        key: disjoint_set.find(
            key
        )
        for key in disjoint_set.parent
    }

    unique_roots = sorted(
        set(
            roots.values()
        )
    )

    component_id_lookup = {
        root: component_id
        for component_id, root in enumerate(
            unique_roots
        )
    }

    node_records = []

    for key, degree in node_degree.items():

        component_id = component_id_lookup[
            roots[
                key
            ]
        ]

        node_records.append({
            "node_key": str(
                key
            ),
            "degree": int(
                degree
            ),
            "component_id": int(
                component_id
            ),
            "geometry": Point(
                key
            ),
        })

    nodes = gpd.GeoDataFrame(
        node_records,
        geometry="geometry",
        crs=TARGET_CRS,
    )

    edge_dataframe = gpd.GeoDataFrame(
        edge_records,
        geometry="geometry",
        crs=TARGET_CRS,
    )

    edge_dataframe[
        "component_id"
    ] = [
        component_id_lookup[
            roots[
                start_key
            ]
        ]
        for start_key in edge_dataframe[
            "start_key"
        ]
    ]

    component_records = []

    for component_id, component_edges in (
        edge_dataframe.groupby(
            "component_id"
        )
    ):

        component_nodes = nodes[
            nodes[
                "component_id"
            ] == component_id
        ]

        component_geometry = shapely.union_all(
            component_edges.geometry.array
        )

        component_records.append({
            "dataset": dataset_name,
            "component_id": int(
                component_id
            ),
            "edge_count": int(
                len(
                    component_edges
                )
            ),
            "node_count": int(
                len(
                    component_nodes
                )
            ),
            "length_m": float(
                component_edges[
                    "length_m"
                ].sum()
            ),
            "geometry": component_geometry,
        })

    components = gpd.GeoDataFrame(
        component_records,
        geometry="geometry",
        crs=TARGET_CRS,
    )

    components = components.sort_values(
        "length_m",
        ascending=False,
    ).reset_index(
        drop=True
    )

    components[
        "length_rank"
    ] = (
        np.arange(
            len(
                components
            )
        )
        + 1
    )

    total_edge_length_m = float(
        edge_dataframe[
            "length_m"
        ].sum()
    )

    largest_component_length_m = float(
        components.iloc[0][
            "length_m"
        ]
    )

    boundary_distance = nodes.geometry.distance(
        region_geometry.boundary
    )

    nodes[
        "boundary_distance_m"
    ] = boundary_distance

    nodes[
        "outside_boundary_exclusion"
    ] = (
        boundary_distance
        > BOUNDARY_EXCLUSION_M
    )

    dangles_all = nodes[
        nodes[
            "degree"
        ] == 1
    ].copy()

    core_nodes = nodes[
        nodes[
            "outside_boundary_exclusion"
        ]
    ].copy()

    dangles_core = core_nodes[
        core_nodes[
            "degree"
        ] == 1
    ].copy()

    dangle_pairs = create_dangle_pair_layer(
        dangles_core
    )

    degree_counts = (
        nodes[
            "degree"
        ]
        .value_counts()
        .sort_index()
    )

    degree_distribution_records = []

    for degree, count in (
        degree_counts.items()
    ):

        degree_distribution_records.append({
            "dataset": dataset_name,
            "degree": int(
                degree
            ),
            "node_count": int(
                count
            ),
            "node_ratio": float(
                count
                / len(
                    nodes
                )
            ),
        })

    degree_distribution = pd.DataFrame(
        degree_distribution_records
    )

    pair_class_counts = (
        dangle_pairs[
            "distance_class"
        ].value_counts()
        if not dangle_pairs.empty
        else pd.Series(
            dtype=int
        )
    )

    area_km2 = (
        region_geometry.area
        / 1_000_000
    )

    metrics = {
        "dataset": dataset_name,

        "topology_node_count": int(
            len(
                nodes
            )
        ),

        "topology_edge_count": int(
            len(
                edge_dataframe
            )
        ),

        "topology_total_length_m": (
            total_edge_length_m
        ),

        "network_density_km_per_km2": (
            total_edge_length_m
            / 1000
            / area_km2
        ),

        "mean_edge_length_m": float(
            edge_dataframe[
                "length_m"
            ].mean()
        ),

        "median_edge_length_m": float(
            edge_dataframe[
                "length_m"
            ].median()
        ),

        "connected_component_count": int(
            len(
                components
            )
        ),

        "largest_component_length_m": (
            largest_component_length_m
        ),

        "largest_component_length_share": (
            largest_component_length_m
            / total_edge_length_m
            if total_edge_length_m > 0
            else np.nan
        ),

        "component_count_lt_100m": int(
            (
                components[
                    "length_m"
                ]
                < SMALL_COMPONENT_THRESHOLD_1_M
            ).sum()
        ),

        "component_count_lt_500m": int(
            (
                components[
                    "length_m"
                ]
                < SMALL_COMPONENT_THRESHOLD_2_M
            ).sum()
        ),

        "degree1_node_count_all": int(
            len(
                dangles_all
            )
        ),

        "degree1_node_ratio_all": float(
            len(
                dangles_all
            )
            / len(
                nodes
            )
        ),

        "core_node_count_excluding_boundary": int(
            len(
                core_nodes
            )
        ),

        "degree1_node_count_excluding_boundary": int(
            len(
                dangles_core
            )
        ),

        "degree1_node_ratio_excluding_boundary": float(
            len(
                dangles_core
            )
            / len(
                core_nodes
            )
            if len(
                core_nodes
            ) > 0
            else np.nan
        ),

        "degree2_node_count": int(
            (
                nodes[
                    "degree"
                ] == 2
            ).sum()
        ),

        "degree2_node_ratio": float(
            (
                nodes[
                    "degree"
                ] == 2
            ).mean()
        ),

        "degree3_node_count": int(
            (
                nodes[
                    "degree"
                ] == 3
            ).sum()
        ),

        "degree3_node_ratio": float(
            (
                nodes[
                    "degree"
                ] == 3
            ).mean()
        ),

        "degree4plus_node_count": int(
            (
                nodes[
                    "degree"
                ] >= 4
            ).sum()
        ),

        "degree4plus_node_ratio": float(
            (
                nodes[
                    "degree"
                ] >= 4
            ).mean()
        ),

        "mean_node_degree": float(
            nodes[
                "degree"
            ].mean()
        ),

        "dangle_nearest_pair_count_le_0_5m": int(
            pair_class_counts.get(
                "≤0.5 m",
                0,
            )
        ),

        "dangle_nearest_pair_count_0_5_to_5m": int(
            pair_class_counts.get(
                "0.5–5 m",
                0,
            )
        ),

        "dangle_nearest_pair_count_5_to_10m": int(
            pair_class_counts.get(
                "5–10 m",
                0,
            )
        ),
    }

    return {
        "metrics": metrics,
        "nodes": nodes,
        "edges": edge_dataframe,
        "components": components,
        "degree_distribution": degree_distribution,
        "dangles_all": dangles_all,
        "dangles_core": dangles_core,
        "dangle_pairs": dangle_pairs,
    }


# ============================================================


## 4. 建筑穿越与缺口提案工具

In [ ]:
# 6. 建筑面和穿越分解
# ============================================================

def prepare_building_union(
    buildings,
    region_geometry,
):
    """
    构建完整建筑面与内缩建筑审计面。
    """

    buildings = buildings.to_crs(
        TARGET_CRS
    ).copy()

    valid_mask = (
        ~buildings.geometry.isna()
        & ~buildings.geometry.is_empty
    )

    buildings = buildings.loc[
        valid_mask
    ].copy()

    buildings = buildings[
        buildings.intersects(
            region_geometry
        )
    ].copy()

    buildings = gpd.clip(
        buildings,
        region_geometry,
    )

    polygons = []

    for geometry in buildings.geometry:

        try:

            fixed = shapely.make_valid(
                geometry
            )

        except Exception:

            try:

                fixed = geometry.buffer(
                    0
                )

            except Exception:

                continue

        polygons.extend(
            iter_polygon_geometries(
                fixed
            )
        )

    if not polygons:

        raise RuntimeError(
            "研究区内没有有效建筑面。"
        )

    building_union = shapely.union_all(
        polygons
    )

    inner_geometries = []

    for polygon in polygons:

        inner = polygon.buffer(
            -BUILDING_INSET_M
        )

        if (
            inner is not None
            and not inner.is_empty
        ):

            inner_geometries.extend(
                iter_polygon_geometries(
                    inner
                )
            )

    if inner_geometries:

        building_inner_union = shapely.union_all(
            inner_geometries
        )

    else:

        building_inner_union = (
            shapely.GeometryCollection()
        )

    return {
        "building_count": int(
            len(
                polygons
            )
        ),
        "building_union": building_union,
        "building_inner_union": (
            building_inner_union
        ),
    }


def calculate_building_crossing_decomposition(
    before_union,
    after_union,
    building_inner_union,
):
    """
    分解：
    - 输入既有穿越
    - 最终总穿越
    - 保留的既有穿越
    - 新增穿越
    - 被消除穿越
    """

    before_crossing = extract_line_geometry(
        before_union.intersection(
            building_inner_union
        )
    )

    after_crossing = extract_line_geometry(
        after_union.intersection(
            building_inner_union
        )
    )

    if before_crossing.is_empty:

        introduced_crossing = (
            after_crossing
        )

        retained_crossing = (
            shapely.GeometryCollection()
        )

    else:

        before_match_area = before_crossing.buffer(
            CROSSING_MATCH_TOLERANCE_M,
            cap_style=2,
            join_style=2,
        )

        introduced_crossing = extract_line_geometry(
            after_crossing.difference(
                before_match_area
            )
        )

        retained_crossing = extract_line_geometry(
            after_crossing.intersection(
                before_match_area
            )
        )

    if after_crossing.is_empty:

        eliminated_crossing = (
            before_crossing
        )

    else:

        after_match_area = after_crossing.buffer(
            CROSSING_MATCH_TOLERANCE_M,
            cap_style=2,
            join_style=2,
        )

        eliminated_crossing = extract_line_geometry(
            before_crossing.difference(
                after_match_area
            )
        )

    return {
        "before_crossing_geometry": (
            before_crossing
        ),
        "after_crossing_geometry": (
            after_crossing
        ),
        "retained_crossing_geometry": (
            retained_crossing
        ),
        "introduced_crossing_geometry": (
            introduced_crossing
        ),
        "eliminated_crossing_geometry": (
            eliminated_crossing
        ),

        "before_crossing_m": safe_length(
            before_crossing
        ),

        "after_crossing_m": safe_length(
            after_crossing
        ),

        "retained_crossing_m": safe_length(
            retained_crossing
        ),

        "introduced_crossing_m": safe_length(
            introduced_crossing
        ),

        "eliminated_crossing_m": safe_length(
            eliminated_crossing
        ),
    }


# ============================================================
# 7. 缺口提案空间去重
# ============================================================

def read_gap_layer(
    path,
    layer_name,
):
    """
    安全读取GPKG图层。
    """

    if not path.exists():

        return make_empty_gdf()

    try:

        result = gpd.read_file(
            path,
            layer=layer_name,
        ).to_crs(
            TARGET_CRS
        )

        return result

    except Exception:

        return make_empty_gdf()


def deduplicate_gap_layer(
    gap_gdf,
    region_geometry,
):
    """
    将缓冲网格中重复出现的缺口提案进行几何去重。
    """

    if gap_gdf is None or gap_gdf.empty:

        return make_empty_gdf(
            columns={
                "connector_length_m": "float64",
            }
        )

    gap_gdf = gap_gdf[
        ~gap_gdf.geometry.isna()
        & ~gap_gdf.geometry.is_empty
    ].copy()

    if gap_gdf.empty:

        return make_empty_gdf(
            columns={
                "connector_length_m": "float64",
            }
        )

    gap_gdf = gap_gdf[
        gap_gdf.intersects(
            region_geometry
        )
    ].copy()

    if gap_gdf.empty:

        return make_empty_gdf(
            columns={
                "connector_length_m": "float64",
            }
        )

    # 只保留连接线中点位于研究区内的提案
    middle_points = gap_gdf.geometry.interpolate(
        0.5,
        normalized=True,
    )

    gap_gdf = gap_gdf[
        middle_points.within(
            region_geometry
        )
        | middle_points.touches(
            region_geometry
        )
    ].copy()

    if gap_gdf.empty:

        return make_empty_gdf(
            columns={
                "connector_length_m": "float64",
            }
        )

    standardized = shapely.set_precision(
        shapely.force_2d(
            gap_gdf.geometry.array
        ),
        GAP_DEDUP_PRECISION_M,
    )

    gap_gdf["_geometry_wkb"] = (
        normalized_wkb(
            standardized
        )
    )

    gap_gdf = (
        gap_gdf
        .drop_duplicates(
            subset="_geometry_wkb"
        )
        .drop(
            columns="_geometry_wkb"
        )
        .reset_index(
            drop=True
        )
    )

    if "connector_length_m" not in gap_gdf.columns:

        gap_gdf[
            "connector_length_m"
        ] = gap_gdf.geometry.length

    return gap_gdf


# ============================================================


## 5. 读取数据并计算基础指标

In [ ]:
# 8. 读取数据
# ============================================================

print("=" * 78)
print("广安门内道路清洗 V13｜全维度质量验证")
print("开始时间：", datetime.datetime.now())


region_hulls = gpd.read_parquet(
    REGION_HULL_PATH
).to_crs(
    TARGET_CRS
)

if REGION_ID in region_hulls.index:
    region_hull = region_hulls.geometry.loc[REGION_ID]
elif len(region_hulls) == 1:
    region_hull = region_hulls.geometry.iloc[0]
    print("边界文件仅含一个要素，已自动使用该要素。")
else:
    raise KeyError(
        f"边界文件中不存在 REGION_ID={REGION_ID}，且文件不止一个要素。"
    )

evaluation_area_km2 = (
    region_hull.area
    / 1_000_000
)


before_source = gpd.read_parquet(
    BEFORE_STREETS_PATH
)

after_source = gpd.read_parquet(
    AFTER_STREETS_PATH
)

buildings_source = gpd.read_parquet(
    BUILDINGS_PATH
)


# ============================================================
# 9. 几何质量与线段统计
# ============================================================

print("\n1/6 正在计算几何质量和线段指标……")

before_prepared = prepare_line_parts(
    before_source,
    region_hull,
    "处理前",
)

after_prepared = prepare_line_parts(
    after_source,
    region_hull,
    "处理后",
)


# ============================================================
# 10. 网络拓扑
# ============================================================

print("2/6 正在建立处理前二维拓扑……")

before_topology = build_topology_metrics(
    before_prepared[
        "line_union"
    ],
    region_hull,
    "处理前",
)

print("3/6 正在建立处理后二维拓扑……")

after_topology = build_topology_metrics(
    after_prepared[
        "line_union"
    ],
    region_hull,
    "处理后",
)


# ============================================================
# 11. 建筑穿越
# ============================================================

print("4/6 正在计算建筑穿越分解……")

building_data = prepare_building_union(
    buildings_source,
    region_hull,
)

crossing_decomposition = (
    calculate_building_crossing_decomposition(
        before_prepared[
            "line_union"
        ],
        after_prepared[
            "line_union"
        ],
        building_data[
            "building_inner_union"
        ],
    )
)


# ============================================================
# 12. 读取过程控制结果
# ============================================================

print("5/6 正在读取V13过程控制指标……")

performance_df = read_optional_csv(
    V13_PERFORMANCE_PATH
)

grid_status_df = read_optional_csv(
    V13_GRID_STATUS_PATH
)

stage_audit_df = read_optional_csv(
    V13_STAGE_AUDIT_PATH
)

crossing_summary_df = read_optional_csv(
    CROSSING_SUMMARY_PATH
)

crossing_stage_df = read_optional_csv(
    CROSSING_STAGE_PATH
)

crossing_grid_df = read_optional_csv(
    CROSSING_GRID_PATH
)


# 缺口提案空间去重
accepted_gap_raw = read_gap_layer(
    V13_GAP_GPKG_PATH,
    "accepted_close_gaps",
)

rejected_change_raw = read_gap_layer(
    V13_GAP_GPKG_PATH,
    "rejected_changes",
)


if (
    not rejected_change_raw.empty
    and "stage"
    in rejected_change_raw.columns
):

    rejected_gap_raw = (
        rejected_change_raw[
            rejected_change_raw[
                "stage"
            ].astype(str)
            == "close_gaps"
        ]
        .copy()
    )

else:

    rejected_gap_raw = (
        rejected_change_raw
    )


accepted_gap_unique = deduplicate_gap_layer(
    accepted_gap_raw,
    region_hull,
)

rejected_gap_unique = deduplicate_gap_layer(
    rejected_gap_raw,
    region_hull,
)


# ============================================================
# 13. 汇总基础字典
# ============================================================

before_quality = before_prepared[
    "quality"
]

after_quality = after_prepared[
    "quality"
]

before_network = before_topology[
    "metrics"
]

after_network = after_topology[
    "metrics"
]


# 建筑保护成效
potential_new_crossing_m = first_numeric(
    crossing_summary_df,
    [
        "检测到的潜在新增穿越长度_m",
    ],
)

prevented_new_crossing_m = first_numeric(
    crossing_summary_df,
    [
        "被保护机制阻止的新增穿越长度_m",
    ],
)

residual_new_crossing_m = first_numeric(
    crossing_summary_df,
    [
        "最终残余新增穿越长度_m",
    ],
    default=crossing_decomposition[
        "introduced_crossing_m"
    ],
)

crossing_prevention_rate = first_numeric(
    crossing_summary_df,
    [
        "新增穿越阻止率",
    ],
)


# 运行指标
run_time_seconds = first_numeric(
    performance_df,
    [
        "run_time_seconds",
    ],
)

peak_memory_mb = first_numeric(
    performance_df,
    [
        "peak_memory_mb",
    ],
)

processing_success_rate = first_numeric(
    performance_df,
    [
        "processing_success_rate",
    ],
)

generated_grid_count = first_numeric(
    performance_df,
    [
        "generated_grid_count",
    ],
)

submitted_grid_count = first_numeric(
    performance_df,
    [
        "submitted_grid_count",
    ],
)

successful_grid_count = first_numeric(
    performance_df,
    [
        "successful_grid_count",
    ],
)

error_grid_count = first_numeric(
    performance_df,
    [
        "error_grid_count",
    ],
)


if not grid_status_df.empty:

    if "status" in grid_status_df.columns:

        successful_grid_count = int(
            (
                grid_status_df[
                    "status"
                ] == "success"
            ).sum()
        )

        error_grid_count = int(
            (
                grid_status_df[
                    "status"
                ] == "error"
            ).sum()
        )

        submitted_grid_count = int(
            len(
                grid_status_df
            )
        )

        processing_success_rate = (
            successful_grid_count
            / submitted_grid_count
            if submitted_grid_count > 0
            else np.nan
        )


# 阶段回退次数
if (
    not stage_audit_df.empty
    and "rollback"
    in stage_audit_df.columns
):

    rollback_count = int(
        pd.to_numeric(
            stage_audit_df[
                "rollback"
            ],
            errors="coerce",
        )
        .fillna(0)
        .astype(bool)
        .sum()
    )

else:

    rollback_count = 0


# 缺口指标
accepted_gap_raw_count = int(
    len(
        accepted_gap_raw
    )
)

accepted_gap_unique_count = int(
    len(
        accepted_gap_unique
    )
)

accepted_gap_unique_length_m = float(
    accepted_gap_unique[
        "connector_length_m"
    ].sum()
    if (
        not accepted_gap_unique.empty
        and "connector_length_m"
        in accepted_gap_unique.columns
    )
    else 0.0
)

rejected_gap_raw_count = int(
    len(
        rejected_gap_raw
    )
)

rejected_gap_unique_count = int(
    len(
        rejected_gap_unique
    )
)

rejected_gap_unique_length_m = float(
    rejected_gap_unique[
        "connector_length_m"
    ].sum()
    if (
        not rejected_gap_unique.empty
        and "connector_length_m"
        in rejected_gap_unique.columns
    )
    else 0.0
)


# ============================================================


## 6. 构建处理前后比较表与过程控制指标

In [ ]:
# 14. 处理前后比较表
# ============================================================

comparison_rows = []


def add_comparison_metric(
    category,
    metric,
    before_value,
    after_value,
    unit,
    note="",
):
    """
    添加一项处理前后比较指标。
    """

    before_value = (
        float(before_value)
        if before_value is not None
        else np.nan
    )

    after_value = (
        float(after_value)
        if after_value is not None
        else np.nan
    )

    difference = (
        after_value
        - before_value
        if (
            np.isfinite(
                before_value
            )
            and np.isfinite(
                after_value
            )
        )
        else np.nan
    )

    change_rate = percent_change(
        before_value,
        after_value,
    )

    comparison_rows.append({
        "维度": category,
        "指标": metric,
        "处理前": before_value,
        "处理后": after_value,
        "绝对变化": difference,
        "变化率": change_rate,
        "单位": unit,
        "说明": note,
    })


# ------------------------------------------------------------
# 几何与规模
# ------------------------------------------------------------

add_comparison_metric(
    "几何与规模",
    "研究区内源要素数",
    before_quality[
        "intersecting_source_feature_count"
    ],
    after_quality[
        "intersecting_source_feature_count"
    ],
    "条",
)

add_comparison_metric(
    "几何与规模",
    "有效线部件数",
    before_quality[
        "line_part_count"
    ],
    after_quality[
        "line_part_count"
    ],
    "条",
)

add_comparison_metric(
    "几何与规模",
    "唯一线部件数",
    before_quality[
        "unique_line_part_count"
    ],
    after_quality[
        "unique_line_part_count"
    ],
    "条",
)

add_comparison_metric(
    "几何与规模",
    "道路总长度",
    before_quality[
        "union_length_m"
    ] / 1000,
    after_quality[
        "union_length_m"
    ] / 1000,
    "km",
    "对重叠线进行唯一溶解后的长度",
)

add_comparison_metric(
    "几何与规模",
    "网络密度",
    before_network[
        "network_density_km_per_km2"
    ],
    after_network[
        "network_density_km_per_km2"
    ],
    "km/km²",
)

add_comparison_metric(
    "几何与规模",
    "平均拓扑边长度",
    before_network[
        "mean_edge_length_m"
    ],
    after_network[
        "mean_edge_length_m"
    ],
    "m",
)

add_comparison_metric(
    "几何与规模",
    "中位拓扑边长度",
    before_network[
        "median_edge_length_m"
    ],
    after_network[
        "median_edge_length_m"
    ],
    "m",
)


# ------------------------------------------------------------
# 几何有效性
# ------------------------------------------------------------

for metric_name, metric_key in [
    (
        "空几何数",
        "empty_geometry_count",
    ),
    (
        "缺失几何数",
        "missing_geometry_count",
    ),
    (
        "无效几何数",
        "invalid_geometry_count",
    ),
    (
        "非线型几何数",
        "non_line_geometry_count",
    ),
    (
        "非简单线数",
        "non_simple_line_count",
    ),
    (
        "完全重复冗余副本",
        "exact_duplicate_redundant_count",
    ),
]:

    add_comparison_metric(
        "几何有效性",
        metric_name,
        before_quality[
            metric_key
        ],
        after_quality[
            metric_key
        ],
        "条",
    )


add_comparison_metric(
    "几何有效性",
    "局部重叠冗余长度",
    before_quality[
        "overlap_redundant_length_m"
    ],
    after_quality[
        "overlap_redundant_length_m"
    ],
    "m",
    "唯一线部件总长度减去union_all后的唯一线长",
)


# ------------------------------------------------------------
# 短线与碎片
# ------------------------------------------------------------

for metric_name, metric_key in [
    (
        "长度≤0.5 m线段",
        "line_count_le_0_5m",
    ),
    (
        "长度0.5–5 m线段",
        "line_count_0_5_to_5m",
    ),
    (
        "长度5–10 m线段",
        "line_count_5_to_10m",
    ),
]:

    add_comparison_metric(
        "短线与碎片",
        metric_name,
        before_quality[
            metric_key
        ],
        after_quality[
            metric_key
        ],
        "条",
    )


# ------------------------------------------------------------
# 拓扑规模
# ------------------------------------------------------------

for metric_name, metric_key, unit in [
    (
        "拓扑节点数",
        "topology_node_count",
        "个",
    ),
    (
        "拓扑边数",
        "topology_edge_count",
        "条",
    ),
    (
        "连通分量数",
        "connected_component_count",
        "个",
    ),
    (
        "长度小于100 m的连通分量",
        "component_count_lt_100m",
        "个",
    ),
    (
        "长度小于500 m的连通分量",
        "component_count_lt_500m",
        "个",
    ),
]:

    add_comparison_metric(
        "网络连通性",
        metric_name,
        before_network[
            metric_key
        ],
        after_network[
            metric_key
        ],
        unit,
    )


add_comparison_metric(
    "网络连通性",
    "最大连通分量长度占比",
    before_network[
        "largest_component_length_share"
    ],
    after_network[
        "largest_component_length_share"
    ],
    "比例",
)


# ------------------------------------------------------------
# 节点度数
# ------------------------------------------------------------

for metric_name, metric_key, unit in [
    (
        "度为1节点数",
        "degree1_node_count_all",
        "个",
    ),
    (
        "度为1节点比例",
        "degree1_node_ratio_all",
        "比例",
    ),
    (
        "排除边界15 m后的度为1节点数",
        "degree1_node_count_excluding_boundary",
        "个",
    ),
    (
        "排除边界15 m后的度为1节点比例",
        "degree1_node_ratio_excluding_boundary",
        "比例",
    ),
    (
        "度为2节点比例",
        "degree2_node_ratio",
        "比例",
    ),
    (
        "度为3节点比例",
        "degree3_node_ratio",
        "比例",
    ),
    (
        "度为4及以上节点比例",
        "degree4plus_node_ratio",
        "比例",
    ),
    (
        "平均节点度数",
        "mean_node_degree",
        "度",
    ),
]:

    add_comparison_metric(
        "节点结构",
        metric_name,
        before_network[
            metric_key
        ],
        after_network[
            metric_key
        ],
        unit,
    )


# ------------------------------------------------------------
# 残余近距离端点
# ------------------------------------------------------------

for metric_name, metric_key in [
    (
        "最近距离≤0.5 m的度1节点对",
        "dangle_nearest_pair_count_le_0_5m",
    ),
    (
        "最近距离0.5–5 m的度1节点对",
        "dangle_nearest_pair_count_0_5_to_5m",
    ),
    (
        "最近距离5–10 m的度1节点对",
        "dangle_nearest_pair_count_5_to_10m",
    ),
]:

    add_comparison_metric(
        "悬挂节点与潜在缺口",
        metric_name,
        before_network[
            metric_key
        ],
        after_network[
            metric_key
        ],
        "对",
        "仅统计排除研究边界15 m后的度1节点",
    )


comparison_report = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 15. 过程控制指标
# ============================================================

process_rows = []


def add_process_metric(
    category,
    metric,
    value,
    unit,
    note="",
):
    """
    添加单值过程控制指标。
    """

    process_rows.append({
        "维度": category,
        "指标": metric,
        "数值": value,
        "单位": unit,
        "说明": note,
    })


# ------------------------------------------------------------
# 建筑穿越背景
# ------------------------------------------------------------

add_process_metric(
    "建筑穿越背景",
    "输入既有建筑穿越长度",
    crossing_decomposition[
        "before_crossing_m"
    ],
    "m",
    "作为数据背景，不直接认定为错误",
)

add_process_metric(
    "建筑穿越背景",
    "最终总建筑穿越长度",
    crossing_decomposition[
        "after_crossing_m"
    ],
    "m",
)

add_process_metric(
    "建筑穿越背景",
    "最终保留的既有建筑穿越长度",
    crossing_decomposition[
        "retained_crossing_m"
    ],
    "m",
)

add_process_metric(
    "建筑穿越背景",
    "处理过程中消除的既有建筑穿越长度",
    crossing_decomposition[
        "eliminated_crossing_m"
    ],
    "m",
)


# ------------------------------------------------------------
# 建筑新增穿越保护
# ------------------------------------------------------------

add_process_metric(
    "建筑穿越保护",
    "检测到的潜在新增建筑穿越长度",
    potential_new_crossing_m,
    "m",
)

add_process_metric(
    "建筑穿越保护",
    "被保护机制阻止的新增建筑穿越长度",
    prevented_new_crossing_m,
    "m",
)

add_process_metric(
    "建筑穿越保护",
    "最终残余新增建筑穿越长度",
    residual_new_crossing_m,
    "m",
)

add_process_metric(
    "建筑穿越保护",
    "新增建筑穿越阻止率",
    crossing_prevention_rate,
    "比例",
)


# ------------------------------------------------------------
# 分阶段阻止长度
# ------------------------------------------------------------

if not crossing_stage_df.empty:

    for _, row in crossing_stage_df.iterrows():

        stage_name = row.get(
            "阶段",
            "未知阶段",
        )

        prevented_length = row.get(
            "被阻止穿越长度_m",
            np.nan,
        )

        add_process_metric(
            "分阶段建筑保护",
            f"{stage_name}阻止的新增穿越",
            prevented_length,
            "m",
            str(
                row.get(
                    "保护方式",
                    "",
                )
            ),
        )


# ------------------------------------------------------------
# 缺口修复
# ------------------------------------------------------------

add_process_metric(
    "缺口修复",
    "缓冲网格接受提案数",
    accepted_gap_raw_count,
    "组",
    "包含相邻网格缓冲区中的重复提案",
)

add_process_metric(
    "缺口修复",
    "空间去重后的接受提案数",
    accepted_gap_unique_count,
    "组",
    "按1 mm精度标准化几何后去重",
)

add_process_metric(
    "缺口修复",
    "去重后接受提案的端点吸附距离合计",
    accepted_gap_unique_length_m,
    "m",
    "不是最终新增道路长度",
)

add_process_metric(
    "缺口修复",
    "缓冲网格拒绝提案数",
    rejected_gap_raw_count,
    "组",
)

add_process_metric(
    "缺口修复",
    "空间去重后的拒绝提案数",
    rejected_gap_unique_count,
    "组",
)

add_process_metric(
    "缺口修复",
    "去重后拒绝提案连接长度",
    rejected_gap_unique_length_m,
    "m",
)


# ------------------------------------------------------------
# 运行稳定性
# ------------------------------------------------------------

add_process_metric(
    "运行稳定性",
    "生成网格数",
    generated_grid_count,
    "个",
)

add_process_metric(
    "运行稳定性",
    "提交任务数",
    submitted_grid_count,
    "个",
)

add_process_metric(
    "运行稳定性",
    "成功网格数",
    successful_grid_count,
    "个",
)

add_process_metric(
    "运行稳定性",
    "失败网格数",
    error_grid_count,
    "个",
)

add_process_metric(
    "运行稳定性",
    "网格成功率",
    processing_success_rate,
    "比例",
)

add_process_metric(
    "运行稳定性",
    "阶段回退次数",
    rollback_count,
    "次",
)

add_process_metric(
    "运行稳定性",
    "完整运行时间",
    run_time_seconds,
    "s",
)

add_process_metric(
    "运行稳定性",
    "父子进程峰值内存",
    peak_memory_mb,
    "MB",
)


process_report = pd.DataFrame(
    process_rows
)


# ============================================================
# 16. 全指标长表
# ============================================================

all_metric_rows = []


for _, row in comparison_report.iterrows():

    all_metric_rows.append({
        "指标类型": "处理前后比较",
        "维度": row[
            "维度"
        ],
        "指标": row[
            "指标"
        ],
        "处理前": row[
            "处理前"
        ],
        "处理后": row[
            "处理后"
        ],
        "绝对变化": row[
            "绝对变化"
        ],
        "变化率": row[
            "变化率"
        ],
        "单值": np.nan,
        "单位": row[
            "单位"
        ],
        "说明": row[
            "说明"
        ],
    })


for _, row in process_report.iterrows():

    all_metric_rows.append({
        "指标类型": "过程控制指标",
        "维度": row[
            "维度"
        ],
        "指标": row[
            "指标"
        ],
        "处理前": np.nan,
        "处理后": np.nan,
        "绝对变化": np.nan,
        "变化率": np.nan,
        "单值": row[
            "数值"
        ],
        "单位": row[
            "单位"
        ],
        "说明": row[
            "说明"
        ],
    })


all_metrics_report = pd.DataFrame(
    all_metric_rows
)


# ============================================================
# 17. 节点度数与连通分量表
# ============================================================

degree_distribution_report = pd.concat([
    before_topology[
        "degree_distribution"
    ],
    after_topology[
        "degree_distribution"
    ],
], ignore_index=True)


component_report = pd.concat([
    before_topology[
        "components"
    ][[
        "dataset",
        "component_id",
        "length_rank",
        "edge_count",
        "node_count",
        "length_m",
    ]],

    after_topology[
        "components"
    ][[
        "dataset",
        "component_id",
        "length_rank",
        "edge_count",
        "node_count",
        "length_m",
    ]],
], ignore_index=True)


# ============================================================
# 18. 宽表汇总
# ============================================================

summary_wide = pd.DataFrame([
    {
        "评价范围_km2": (
            evaluation_area_km2
        ),

        "坐标系": TARGET_CRS,

        "处理前道路要素数": (
            before_quality[
                "intersecting_source_feature_count"
            ]
        ),

        "处理后道路要素数": (
            after_quality[
                "intersecting_source_feature_count"
            ]
        ),

        "处理前道路长度_km": (
            before_quality[
                "union_length_m"
            ]
            / 1000
        ),

        "处理后道路长度_km": (
            after_quality[
                "union_length_m"
            ]
            / 1000
        ),

        "道路长度变化率": (
            percent_change(
                before_quality[
                    "union_length_m"
                ],
                after_quality[
                    "union_length_m"
                ],
            )
        ),

        "处理前拓扑节点数": (
            before_network[
                "topology_node_count"
            ]
        ),

        "处理后拓扑节点数": (
            after_network[
                "topology_node_count"
            ]
        ),

        "处理前拓扑边数": (
            before_network[
                "topology_edge_count"
            ]
        ),

        "处理后拓扑边数": (
            after_network[
                "topology_edge_count"
            ]
        ),

        "处理前连通分量数": (
            before_network[
                "connected_component_count"
            ]
        ),

        "处理后连通分量数": (
            after_network[
                "connected_component_count"
            ]
        ),

        "处理前最大分量占比": (
            before_network[
                "largest_component_length_share"
            ]
        ),

        "处理后最大分量占比": (
            after_network[
                "largest_component_length_share"
            ]
        ),

        "处理前边界外度1节点比例": (
            before_network[
                "degree1_node_ratio_excluding_boundary"
            ]
        ),

        "处理后边界外度1节点比例": (
            after_network[
                "degree1_node_ratio_excluding_boundary"
            ]
        ),

        "输入既有建筑穿越_m": (
            crossing_decomposition[
                "before_crossing_m"
            ]
        ),

        "检测到的潜在新增穿越_m": (
            potential_new_crossing_m
        ),

        "被阻止的新增穿越_m": (
            prevented_new_crossing_m
        ),

        "最终残余新增穿越_m": (
            residual_new_crossing_m
        ),

        "新增穿越阻止率": (
            crossing_prevention_rate
        ),

        "缓冲网格接受缺口提案数": (
            accepted_gap_raw_count
        ),

        "空间去重后接受缺口提案数": (
            accepted_gap_unique_count
        ),

        "空间去重后吸附距离_m": (
            accepted_gap_unique_length_m
        ),

        "网格成功率": (
            processing_success_rate
        ),

        "运行时间_s": (
            run_time_seconds
        ),

        "峰值内存_MB": (
            peak_memory_mb
        ),
    }
])


# ============================================================


## 7. 保存统计表与空间检查图层

In [ ]:
# 19. 保存CSV
# ============================================================

comparison_report.to_csv(
    OUTPUT_COMPARISON_CSV,
    index=False,
    encoding="utf-8-sig",
)

process_report.to_csv(
    OUTPUT_PROCESS_CSV,
    index=False,
    encoding="utf-8-sig",
)

all_metrics_report.to_csv(
    OUTPUT_ALL_METRICS_CSV,
    index=False,
    encoding="utf-8-sig",
)

degree_distribution_report.to_csv(
    OUTPUT_DEGREE_CSV,
    index=False,
    encoding="utf-8-sig",
)

component_report.to_csv(
    OUTPUT_COMPONENT_CSV,
    index=False,
    encoding="utf-8-sig",
)

summary_wide.to_csv(
    OUTPUT_SUMMARY_WIDE_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 20. 导出质量问题空间图层
# ============================================================

if OUTPUT_GPKG.exists():

    OUTPUT_GPKG.unlink()


layers_to_export = []


def append_layer(
    layer_name,
    gdf,
    columns=None,
):
    """
    将非空图层加入待导出列表。
    """

    if (
        gdf is None
        or gdf.empty
    ):

        return

    export_gdf = gdf.copy()

    if columns is not None:

        existing_columns = [
            column
            for column in columns
            if column in export_gdf.columns
        ]

        export_gdf = export_gdf[
            existing_columns
            + ["geometry"]
        ]

    layers_to_export.append(
        (
            layer_name,
            export_gdf,
        )
    )


# 度1节点
append_layer(
    "before_dangles_core",
    before_topology[
        "dangles_core"
    ],
    [
        "degree",
        "component_id",
        "boundary_distance_m",
    ],
)

append_layer(
    "after_dangles_core",
    after_topology[
        "dangles_core"
    ],
    [
        "degree",
        "component_id",
        "boundary_distance_m",
    ],
)


# 度1节点近邻候选
append_layer(
    "before_dangle_pairs",
    before_topology[
        "dangle_pairs"
    ],
    [
        "pair_id",
        "distance_m",
        "distance_class",
    ],
)

append_layer(
    "after_dangle_pairs",
    after_topology[
        "dangle_pairs"
    ],
    [
        "pair_id",
        "distance_m",
        "distance_class",
    ],
)


# 短线
before_short_lines = (
    before_prepared[
        "line_parts"
    ][
        before_prepared[
            "line_parts"
        ].geometry.length
        <= SHORT_LINE_THRESHOLD_2_M
    ]
    .copy()
)

after_short_lines = (
    after_prepared[
        "line_parts"
    ][
        after_prepared[
            "line_parts"
        ].geometry.length
        <= SHORT_LINE_THRESHOLD_2_M
    ]
    .copy()
)


append_layer(
    "before_short_lines_le_5m",
    before_short_lines,
    [
        "length_m",
    ],
)

append_layer(
    "after_short_lines_le_5m",
    after_short_lines,
    [
        "length_m",
    ],
)


# 小型连通分量
before_small_components = (
    before_topology[
        "components"
    ][
        before_topology[
            "components"
        ][
            "length_m"
        ]
        < SMALL_COMPONENT_THRESHOLD_2_M
    ]
    .copy()
)

after_small_components = (
    after_topology[
        "components"
    ][
        after_topology[
            "components"
        ][
            "length_m"
        ]
        < SMALL_COMPONENT_THRESHOLD_2_M
    ]
    .copy()
)


append_layer(
    "before_components_lt_500m",
    before_small_components,
    [
        "component_id",
        "length_rank",
        "edge_count",
        "node_count",
        "length_m",
    ],
)

append_layer(
    "after_components_lt_500m",
    after_small_components,
    [
        "component_id",
        "length_rank",
        "edge_count",
        "node_count",
        "length_m",
    ],
)


# 建筑穿越
for layer_name, geometry in [
    (
        "input_existing_crossing",
        crossing_decomposition[
            "before_crossing_geometry"
        ],
    ),
    (
        "final_total_crossing",
        crossing_decomposition[
            "after_crossing_geometry"
        ],
    ),
    (
        "final_introduced_crossing",
        crossing_decomposition[
            "introduced_crossing_geometry"
        ],
    ),
    (
        "eliminated_existing_crossing",
        crossing_decomposition[
            "eliminated_crossing_geometry"
        ],
    ),
]:

    layer_gdf = geometry_to_line_gdf(
        geometry,
        minimum_length_m=0.01,
    )

    append_layer(
        layer_name,
        layer_gdf,
        [
            "part_id",
            "length_m",
        ],
    )


# 缺口提案
append_layer(
    "accepted_gaps_unique",
    accepted_gap_unique,
)

append_layer(
    "rejected_gaps_unique",
    rejected_gap_unique,
)


gpkg_written = False

for layer_name, layer_gdf in (
    layers_to_export
):

    layer_gdf.to_file(
        OUTPUT_GPKG,
        layer=layer_name,
        driver="GPKG",
        mode=(
            "a"
            if gpkg_written
            else "w"
        ),
    )

    gpkg_written = True


# ============================================================


## 8. 核心指标总览与文本报告

In [ ]:
# 21. 绘制核心指标总览
# ============================================================

print("6/6 正在绘制核心指标总览……")

short_before = (
    before_quality["line_count_le_0_5m"]
    + before_quality["line_count_0_5_to_5m"]
)
short_after = (
    after_quality["line_count_le_0_5m"]
    + after_quality["line_count_0_5_to_5m"]
)

metric_cards = [
    ("有效线部件", before_quality["line_part_count"], after_quality["line_part_count"], "条", "处理前", "处理后"),
    ("道路总长度", before_quality["union_length_m"] / 1000, after_quality["union_length_m"] / 1000, "km", "处理前", "处理后"),
    ("长度≤5 m线段", short_before, short_after, "条", "处理前", "处理后"),
    ("拓扑节点", before_network["topology_node_count"], after_network["topology_node_count"], "个", "处理前", "处理后"),
    ("拓扑边", before_network["topology_edge_count"], after_network["topology_edge_count"], "条", "处理前", "处理后"),
    ("最大连通分量占比", before_network["largest_component_length_share"] * 100, after_network["largest_component_length_share"] * 100, "%", "处理前", "处理后"),
    ("边界外度1节点", before_network["degree1_node_count_excluding_boundary"], after_network["degree1_node_count_excluding_boundary"], "个", "处理前", "处理后"),
]

if np.isfinite(potential_new_crossing_m):
    metric_cards.append((
        "新增建筑穿越",
        potential_new_crossing_m,
        residual_new_crossing_m,
        "m",
        "潜在新增",
        "最终残余",
    ))
else:
    metric_cards.append((
        "最终残余新增建筑穿越",
        0.0,
        residual_new_crossing_m,
        "m",
        "基准",
        "最终残余",
    ))

fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=160)
axes = axes.ravel()

for ax, (title, before_value, after_value, unit, before_label, after_label) in zip(axes, metric_cards):
    values = [float(before_value), float(after_value)]
    labels = [before_label, after_label]
    bars = ax.bar(labels, values)
    vmax = max(values) if max(values) > 0 else 1.0
    ax.set_ylim(0, vmax * 1.25)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(unit)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    for bar, value in zip(bars, values):
        if unit in {"km", "%", "m"}:
            label = f"{value:,.2f}"
        else:
            label = f"{value:,.0f}"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + vmax * 0.03,
            label,
            ha="center",
            va="bottom",
            fontsize=9,
        )

fig.suptitle(
    "道路清洗示例｜核心质量指标总览",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)
fig.text(
    0.5,
    0.015,
    "道路长度和线部件数量下降与平行线归并、几何规整有关，需结合连通性、短线和空间冲突指标综合解释。",
    ha="center",
    fontsize=9,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.94])
fig.savefig(
    OUTPUT_DASHBOARD_PNG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()


# ============================================================
# 22. 输出TXT报告
# ============================================================

report_lines = [
    "广安门内道路清洗 V13｜全维度质量验证",
    "=" * 72,
    "",
    "评价设置",
    f"- 坐标系：{TARGET_CRS}",
    f"- 区域编号：{REGION_ID}",
    f"- 评价范围：{evaluation_area_km2:.3f} km²",
    f"- 边界排除距离：{BOUNDARY_EXCLUSION_M:.2f} m",
    f"- 建筑审计内缩：{BUILDING_INSET_M:.2f} m",
    f"- 建筑穿越匹配容差：{CROSSING_MATCH_TOLERANCE_M:.2f} m",
    "",
    "道路规模",
    (
        "- 道路总长度："
        f"{before_quality['union_length_m'] / 1000:.2f}"
        " km → "
        f"{after_quality['union_length_m'] / 1000:.2f}"
        " km"
    ),
    (
        "- 有效线部件数："
        f"{before_quality['line_part_count']:,}"
        " → "
        f"{after_quality['line_part_count']:,}"
    ),
    (
        "- 完全重复冗余副本："
        f"{before_quality['exact_duplicate_redundant_count']:,}"
        " → "
        f"{after_quality['exact_duplicate_redundant_count']:,}"
    ),
    (
        "- 长度≤0.5 m线段："
        f"{before_quality['line_count_le_0_5m']:,}"
        " → "
        f"{after_quality['line_count_le_0_5m']:,}"
    ),
    (
        "- 长度0.5–5 m线段："
        f"{before_quality['line_count_0_5_to_5m']:,}"
        " → "
        f"{after_quality['line_count_0_5_to_5m']:,}"
    ),
    "",
    "网络拓扑",
    (
        "- 拓扑节点："
        f"{before_network['topology_node_count']:,}"
        " → "
        f"{after_network['topology_node_count']:,}"
    ),
    (
        "- 拓扑边："
        f"{before_network['topology_edge_count']:,}"
        " → "
        f"{after_network['topology_edge_count']:,}"
    ),
    (
        "- 连通分量："
        f"{before_network['connected_component_count']:,}"
        " → "
        f"{after_network['connected_component_count']:,}"
    ),
    (
        "- 最大连通分量长度占比："
        f"{before_network['largest_component_length_share']:.2%}"
        " → "
        f"{after_network['largest_component_length_share']:.2%}"
    ),
    (
        "- 排除边界后的度为1节点比例："
        f"{before_network['degree1_node_ratio_excluding_boundary']:.2%}"
        " → "
        f"{after_network['degree1_node_ratio_excluding_boundary']:.2%}"
    ),
    "",
    "建筑穿越保护",
    (
        "- 输入既有建筑穿越："
        f"{crossing_decomposition['before_crossing_m']:.2f} m"
    ),
    (
        "- 检测到的潜在新增建筑穿越："
        f"{potential_new_crossing_m:.2f} m"
        if np.isfinite(
            potential_new_crossing_m
        )
        else "- 检测到的潜在新增建筑穿越：未读取"
    ),
    (
        "- 被保护机制阻止的新增穿越："
        f"{prevented_new_crossing_m:.2f} m"
        if np.isfinite(
            prevented_new_crossing_m
        )
        else "- 被保护机制阻止的新增穿越：未读取"
    ),
    (
        "- 最终残余新增穿越："
        f"{residual_new_crossing_m:.6f} m"
    ),
    (
        "- 新增穿越阻止率："
        f"{crossing_prevention_rate:.2%}"
        if np.isfinite(
            crossing_prevention_rate
        )
        else "- 新增穿越阻止率：未读取"
    ),
    "",
    "缺口修复",
    (
        "- 缓冲网格接受提案："
        f"{accepted_gap_raw_count:,}组"
    ),
    (
        "- 空间去重后接受提案："
        f"{accepted_gap_unique_count:,}组"
    ),
    (
        "- 去重后端点吸附距离合计："
        f"{accepted_gap_unique_length_m:.2f} m"
    ),
    (
        "- 空间去重后拒绝提案："
        f"{rejected_gap_unique_count:,}组"
    ),
    "",
    "运行稳定性",
    (
        "- 网格成功率："
        f"{processing_success_rate:.2%}"
        if np.isfinite(
            processing_success_rate
        )
        else "- 网格成功率：未读取"
    ),
    (
        "- 完整运行时间："
        f"{run_time_seconds:.2f} s"
        if np.isfinite(
            run_time_seconds
        )
        else "- 完整运行时间：未读取"
    ),
    (
        "- 父子进程峰值内存："
        f"{peak_memory_mb:.2f} MB"
        if np.isfinite(
            peak_memory_mb
        )
        else "- 父子进程峰值内存：未读取"
    ),
    "",
    "解释限制",
    "- 网络拓扑为二维平面拓扑，不区分桥梁、隧道和不同高程。",
    "- 输入既有建筑穿越不直接视为错误，其中可能包含过街楼等真实情况。",
    "- 缺口提案按几何空间去重，但仍建议进行人工地图复核。",
    "- 道路长度和要素数量下降不能单独等同于质量提升。",
]


OUTPUT_REPORT_TXT.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


# ============================================================
# 23. 控制台和表格显示
# ============================================================

print("\n" + "=" * 78)
print("全维度质量验证完成")

print(
    "道路长度：",
    (
        f"{before_quality['union_length_m'] / 1000:.2f} km"
        " → "
        f"{after_quality['union_length_m'] / 1000:.2f} km"
    ),
)

print(
    "连通分量：",
    (
        f"{before_network['connected_component_count']}"
        " → "
        f"{after_network['connected_component_count']}"
    ),
)

print(
    "最大连通分量占比：",
    (
        f"{before_network['largest_component_length_share']:.2%}"
        " → "
        f"{after_network['largest_component_length_share']:.2%}"
    ),
)

print(
    "排除边界后的度1节点比例：",
    (
        f"{before_network['degree1_node_ratio_excluding_boundary']:.2%}"
        " → "
        f"{after_network['degree1_node_ratio_excluding_boundary']:.2%}"
    ),
)

print(
    "被阻止的新增建筑穿越：",
    (
        f"{prevented_new_crossing_m:.2f} m"
        if np.isfinite(
            prevented_new_crossing_m
        )
        else "未读取"
    ),
)

print(
    "最终残余新增建筑穿越：",
    f"{residual_new_crossing_m:.6f} m",
)

print(
    "空间去重后的安全缺口：",
    f"{accepted_gap_unique_count}组",
)


print("\n输出目录：")
print(
    OUTPUT_DIR
)

print("\n输出文件：")

for output_path in [
    OUTPUT_COMPARISON_CSV,
    OUTPUT_PROCESS_CSV,
    OUTPUT_ALL_METRICS_CSV,
    OUTPUT_DEGREE_CSV,
    OUTPUT_COMPONENT_CSV,
    OUTPUT_SUMMARY_WIDE_CSV,
    OUTPUT_GPKG,
    OUTPUT_REPORT_TXT,
    OUTPUT_DASHBOARD_PNG,
]:

    if output_path.exists():

        print(
            "✓",
            output_path.name,
        )

    else:

        print(
            "—",
            output_path.name,
            "未生成或内容为空",
        )


print("\n处理前后核心指标：")

core_display_metrics = [
    "道路总长度",
    "完全重复冗余副本",
    "长度≤0.5 m线段",
    "长度0.5–5 m线段",
    "拓扑节点数",
    "拓扑边数",
    "连通分量数",
    "最大连通分量长度占比",
    "排除边界15 m后的度为1节点数",
    "排除边界15 m后的度为1节点比例",
    "最近距离0.5–5 m的度1节点对",
]


display(
    comparison_report[
        comparison_report[
            "指标"
        ].isin(
            core_display_metrics
        )
    ].reset_index(
        drop=True
    ).style.format({
        "处理前": "{:,.4f}",
        "处理后": "{:,.4f}",
        "绝对变化": "{:+,.4f}",
        "变化率": "{:+.2%}",
    })
)


print("\n过程控制核心指标：")

display(
    process_report[
        process_report[
            "维度"
        ].isin([
            "建筑穿越保护",
            "缺口修复",
            "运行稳定性",
        ])
    ].reset_index(
        drop=True
    )
)
